# Final Practical: Classification using scikit-learn and MLflow

In this practical, you will use a Scikit-learn dataset (you may use the same as in the Intro MLflow notebook) to train a classification model.

Steps to follow:

    Data exploration: Analyze the provided dataset to understand its structure and content.

    Text preprocessing: Perform text preprocessing tasks such as tokenization and vectorization to prepare the data for modeling.

    Model training: Use Scikit-learn classification algorithms to train a model with the preprocessed data.

    Model evaluation: Evaluate the model’s performance using standard evaluation metrics such as accuracy and recall.

    Metrics logging using MLflow: Use MLflow to log metrics and hyperparameters during training, making it easier to manage and compare experiments.


Note: Since I will not be able to access your MLflow logs, please include screenshots of the MLflow interface in the notebook.

In [1]:
# ====================================================
# =====    Initial installation in notebook      =====
# ===== (then restart kernel and skip this step) =====
# ====================================================

import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

# Define required packages and versions
required = {
    "pandas": ">=2.0.0",
    "numpy": ">=1.25.0",
    "scikit-learn": "==1.8.0",
    "spacy": ">=3.7.0",
    "nltk": ">=3.8.0",
    "mlflow": "==3.10.0",
    "matplotlib": ">=3.8.0",
    "seaborn": ">=0.13.0",
}

to_install = []

for pkg, ver_spec in required.items():
    try:
        installed_version = version(pkg)
        # Simple check: if exact version required, compare directly
        if ver_spec.startswith("==") and installed_version != ver_spec[2:]:
            to_install.append(f"{pkg}{ver_spec}")
        # For >= constraints, compare numerically
        elif ver_spec.startswith(">="):
            from packaging.version import parse as parse_version
            if parse_version(installed_version) < parse_version(ver_spec[2:]):
                to_install.append(f"{pkg}{ver_spec}")
    except PackageNotFoundError:
        # Package missing → install
        to_install.append(f"{pkg}{ver_spec}")

if to_install:
    print("Installing:", to_install)
    subprocess.run([sys.executable, "-m", "pip", "install"] + to_install, check=True)
else:
    print("All packages already satisfy version requirements.")

All packages already satisfy version requirements.


**After running cell above, restart kernel** (shortcut: click <0><0> quickly).

In [3]:
# ====================================
# ==== Downloads of NLP resources ====
# ====================================

# NLTK stopwords
import nltk
nltk.download("stopwords", quiet=True)

# spaCy English model (small)
import spacy
from spacy.cli import download as spacy_download

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError:
    spacy_download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

print("NLP resources downloaded.")

NLP resources downloaded.


In [4]:
# ===========================
# ==== utils.py imports  ====
# ===========================
import sys
from pathlib import Path

# Replace this path with the folder containing notebook and utils.py
notebook_dir = Path("/home/usuario/Documents/KeepCoding/MLOps-LLMOps/mlops-llmops-bd16/Practical")
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from utils import preprocess_reviews, reviews_to_dict
print("Utils imported successfully!")

Utils imported successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/usuario/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
# ===========================
# ===== Library imports =====
# ===========================

# Data processing
import pandas as pd
import numpy as np
import scipy

# Saving
import joblib, pickle, json

# Text processing
from nltk.corpus import stopwords

# ML model and pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score)
from sklearn.inspection import permutation_importance

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scientific computing
from scipy.sparse import hstack

# Model tracking
import mlflow
import mlflow.sklearn

print("Imports done.")

Imports done.


# Part 1. Classification with scikit-learn and MLflow


## 1. Data exploration using scikit-learn


In this section, I briefly explore the dataset to understand its content and structure, following these steps:

1. **Loading the dataset:**

The `Sports_and_Outdoors_5.json` is loaded into a Python dictionary of dictionaries, using the function `reviews_to_dict()` from `utils.py`. Each entry of the main dictionary is an integer from 0 (first review) to n-1 (last review), for a total of n reviews. Each integer entry is a review containing the fields that can be potential features and the target (binary label). The dictionary is also saved to a json file with the prefix `dict_`.


2. **Creating a DataFrame:**

A DataFrame is created from the dataset, using the method `DataFrame.from_dict()` from Pandas, assigning the features and target to columns.

3. **Previewing the first records:**

The first rows of the DataFrame are displayed to gain an initial understanding of the data.

4. **Getting the DataFrame dimensions:**

The shape of the DataFrame is printed to show the number of rows and columns.

5. **Visualizing descriptive statistics:**

Descriptive statistics of the DataFrame are provided, including counts, means, standard deviations, quartiles, and minima and maxima. In the dataset provided, Class 1 (negative sentiment) is clearly the minority class, since the maximum value of `label` is 1 but all other quartiles are 0. In fact, only 6.5% of the data has label 1.

6. **Column names:**

The names of the columns present in the DataFrame are displayed.

**Summary:**

The code presented below serves as a starting point to explore and understand the `Sports_and_Outdoors_5.json` dataset and to perform basic statistical analysis on its numerical features and target variable.


In [6]:
# Load JSON dataset into dictionary using utils function
file = "Sports_and_Outdoors_5.json"
folder = "/home/usuario/Documents/KeepCoding/MLOps-LLMOps/mlops-llmops-bd16/Practical/"
reviews_dict = reviews_to_dict(folder, file)

Dictionary created.
Dictionary saved to: /home/usuario/Documents/KeepCoding/MLOps-LLMOps/mlops-llmops-bd16/Practical/Sports_and_Outdoors_5.json.

To load the file, use this code:

with open('/home/usuario/Documents/KeepCoding/MLOps-LLMOps/mlops-llmops-bd16/Practical/Sports_and_Outdoors_5.json', 'r') as f:
  reviews_dict = json.load(f)


In [7]:
# Convert to DataFrame
df = pd.DataFrame.from_dict(reviews_dict, orient='index')
print("Dataframe created and loaded.")


Dataframe created and loaded.


In [8]:
df.head()

,label,helpful,age,text
0,0,"[0, 0]",178,This came in on time and I am veru happy with ...
1,0,"[1, 1]",902,I had a factory Glock tool that I was using fo...
2,0,"[2, 2]",876,If you don't have a 3/32 punch or would like t...
3,0,"[0, 0]",899,This works no better than any 3/32 punch you w...
4,0,"[0, 0]",456,I purchased this thinking maybe I need a speci...


In [9]:
df.shape

(296337, 4)

In [10]:
df.describe()

,label,age
count,296337.000000,296337.000000
mean,0.146185,485.425924
std,0.353292,409.091571
min,0.000000,0.000000
25%,0.000000,198.000000
50%,0.000000,401.000000
75%,0.000000,605.000000
max,1.000000,4521.000000


## 2. Text preprocessing

In this part, I implement text preprocessing tasks in one large function located in `utils.py` to prepare the data for modeling. The steps in the function are as follows:

1. **Tokenize, lemmatize, lowercase, stop words removal:**

    Using spaCy's built-in functions, each review is tokenized (separated into words, and punctuation and extra spaces are removed), then each token is lemmatized with POS awareness, turned to lowercase, and finally stopwords are removed. The stopwords list used is based on the NLTK default but words that carry contrasting views or polar sentiment are removed from the list so they can remain in the review.

2. **Store review length for use as an additional feature:**

    The length of each review (in number of tokens) is stored to be included as an additional feature, as longer reviews often correlate with negative sentiment. This feature will later be normalized using StandardScaler.

3. **Truncate very long reviews:**

    Once the length of the review is stored, the reviews that are extremely long are truncated to a maximum number of tokens (set as a default to 512), preserving context and concluding sentiment by keeping the first and last tokens of the review and deleting the middle ones.

4. **Reconstruct processed text:**

    The processed text is put back together, appending tokens with single spaces as separators. 

5. **Inform on processing:**

    A short snippet informs on how many reviews have been processed, printing a message every 10,000 reviews.

6. **Add preprocessed review `processedText` and `reviewLength` to DataFrame:**

    The two features to be used in the model are stored as new columns in the DataFrame.

7. **Save preprocessed DataFrame to a pickle file**

    The preprocessed DataFrame is saved to a pickle file.

    To retrieve it, run: `df = pd.read_pickle(folder + "preprocessed_data.pkl")`

   
**Summary:**

The code below runs function `preprocess_reviews()` (imported from `utils.py`), which processes the text of the reviews using NLP techniques so that they can later be used for training the model.

In [11]:
# RUNNING THIS CELL STARTS THE PREPROCESSING (about 45 min)
# Uncomment below if you still want to run it

#preprocess_reviews(df)

# Save df to pickle file for later retrieval
#df.to_pickle(folder+"preprocessed_data.pkl")

In [12]:
# Retrieve preprocessed DataFrame from pickle file
df = pd.read_pickle(folder + "preprocessed_data.pkl")

In [13]:
df.head()

,label,helpful,age,text,reviewLength,processedText
0,0,"[0, 0]",178,This came in on time and I am veru happy with ...,12,come time veru happy use already make take pin...
1,0,"[1, 1]",902,I had a factory Glock tool that I was using fo...,28,factory glock tool use glock 26 27 17 since lo...
2,0,"[2, 2]",876,If you don't have a 3/32 punch or would like t...,24,not 3/32 punch would like one glock bag okay b...
3,0,"[0, 0]",899,This works no better than any 3/32 punch you w...,17,work no well 3/32 punch would find hardware st...
4,0,"[0, 0]",456,I purchased this thinking maybe I need a speci...,29,purchase thinking maybe need special tool easi...


## 3. Model training with scikit-learn

Here I will use scikit-learn to train and validate a classification model for the preprocessed data. I will follow these steps:

1. **Dataset Splitting:**

    The preprocessed dataset is split into training sets X_train_full_raw and y_train_full, and test sets X_test_raw and y_test, using the `train_test_split` function. Only the relevant features are kept, and the split is applied stratifying by `label` due to severe imbalance between the two classes.

2. **Preparing the Test Set:**

    The target set y_test is saved to a CSV file called 'test-target.csv'. The features test set is saved to another CSV file called 'test.csv'.

3. **Training Set Splitting:**

    The training set is further split into training subsets X_train_raw and y_train, and validation subsets X_val_raw and y_val using `train_test_split` again.

4. **Model Configuration:**

    A classification model is configured using the selected model (Logistic Regression) with the chosen parameters.

5. **Preprocessing and training pipeline:**

    A `pipeline` is configured that includes:

* **TF-IDF Vectorization, `reviewLength`Scaling, and combination of both features:**

    A Bag of Words representation, weighted by inverse frequency using TF‑IDF vectorization, is configured, using max_features = 10,000 and n-grams in range (1,3). The bigrams and trigrams will enable the model to distinguish between cases such as "good", "really good" or "really not good".

    The review length field is standardized using StandardScaler, to enable the model to better adjust the weights.

    A `ColumnTransformer` is used to combine the *TF-IDF features* and the *scaled review length* into a single feature matrix which will be fed into the model, allowing the model to use both text content and review length simultaneously.

* **Model Training with Hyperparameter Tuning:**

    `GridSearchCV` is configured to tune hyperparameters using 5-fold cross-validation, selecting the parameters yielding the best F1 score. The full pipeline is displayed in a diagram. Finally, the model is fitted on the X_train subset.

5. **Performance on the validation set:**

    The model is evaluated using the validation set. If not successful, parameters can be modified and the pipeline restarted.

**Summary:**  

The following lines of code thus represent the process of data preparation, model configuration, and training of a classification model using *Logistic Regression* in scikit-learn.


In [14]:
# 1. Dataset Splitting
X_train_full, X_test, y_train_full, y_test = train_test_split(
    df[['processedText','reviewLength']],
    df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=17
)

In [15]:
# 2. Preparing the test set
y_test.to_csv(folder+'test-target.csv', index=False)
X_test.to_csv(folder+'test.csv', index=False)

In [16]:
# 3. Training set Splitting
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=17
)

In [17]:
# TF-IDF Vectorization (with trigrams and 10,000 max features)
cv = TfidfVectorizer(
    ngram_range=(1,3),  # going up to trigrams
    min_df=10,      # ignore tokens that appear less than this number of times (attributed to noise or spelling errors)
    max_df=0.95,       # ignore very high frequency tokens
    max_features=10000 
)

X_train_cv = cv.fit_transform(X_train['processedText'])  # Fit-transform on training data
X_val_cv = cv.transform(X_val['processedText'])
X_test_cv = cv.transform(X_test['processedText'])        # Just transform on testing and validation data

In [18]:
# Standardize the feature 'reviewLength'
# (necessary for LogisticRegression model)

scaler = StandardScaler()

X_train_len = scaler.fit_transform(X_train[['reviewLength']])  # Fit-transform on training data
X_val_len = scaler.transform(X_val[['reviewLength']])        
X_test_len = scaler.transform(X_test[['reviewLength']])        # Just transform on testing and validation data


In [45]:
# 5. Define preprocessing and training pipeline

scoring='precision'
C_params=[5]

# Column transformer for combining features
preprocessor = ColumnTransformer(
    transformers=[
        ('tfidf', TfidfVectorizer(ngram_range=(1,3), max_features=10000, min_df=10, max_df=0.95), 'processedText'),
        ('length', StandardScaler(), ['reviewLength'])
    ]
)

# Pipeline with Logistic Regression
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# Define hyperparameter grid for tuning
param_grid = {
    'classifier__C': C_params,        # Regularization strength
    'classifier__solver': ['lbfgs']   # Fast & stable for large datasets, L2-compatible
}

# Setup GridSearchCV
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=scoring,            # Best F1 score selects best parameter (unbalanced classes)
    cv=5,                    # 5-fold cross-validation for tuning
    n_jobs=-1,               # Use all cores
    verbose=1
)


In [46]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('tfidf', ...), ('length', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [47]:
# Training of the model
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 1 candidates, totalling 5 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=1000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [5], 'classifier__solver': ['lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- 

In [48]:
# Best model and hyperparameters
best_model = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)

Best hyperparameters: {'classifier__C': 5, 'classifier__solver': 'lbfgs'}


In [49]:
# 6. Performance evaluation on the validation set
y_val_pred = best_model.predict(X_val)

print("Validation Metrics:")
print(classification_report(y_val, y_val_pred, digits=4))


Validation Metrics:
              precision    recall  f1-score   support

           0     0.9617    0.8475    0.9010     40483
           1     0.4740    0.8026    0.5960      6931

    accuracy                         0.8409     47414
   macro avg     0.7178    0.8251    0.7485     47414
weighted avg     0.8904    0.8409    0.8564     47414



## 4. Model Evaluation and Retrieval of Parameters

In this section, the previously trained model is evaluated and some standard performance indicators (such as precision and recall) and configuration parameters are retrieved and displayed. The following is a brief description of the steps followed in the code.

1. **Performance evaluation on the training set:**

    The model's performance on the training data is calculated and printed.

2. **Performance evaluation on the test set:**

    The model's performance on the test set is calculated and printed.

3. **Retrieval and storage of the model's parameters and metrics:**

    The model parameters are obtained using the `get_params` function. This provides detailed information about the model's configuration.

**Summary:**

The code below thus focuses on evaluating the model's performance on the training and test sets. It also retrieves and stores detailed information about the model's configuration.

In [50]:
# 1. Performance evaluation on the training set
y_train_pred = best_model.predict(X_train)

accuracy_train  = accuracy_score(y_train, y_train_pred)
precision_train = precision_score(y_train, y_train_pred)
recall_train    = recall_score(y_train, y_train_pred)
f1_train        = f1_score(y_train, y_train_pred)

print("Train Metrics:")
print(classification_report(y_train, y_train_pred, digits=4))

Train Metrics:
              precision    recall  f1-score   support

           0     0.9772    0.8656    0.9181    161930
           1     0.5292    0.8822    0.6616     27725

    accuracy                         0.8681    189655
   macro avg     0.7532    0.8739    0.7898    189655
weighted avg     0.9117    0.8681    0.8806    189655



In [51]:
# 2. Performance evaluation on the test set
y_test_pred = best_model.predict(X_test)

accuracy_test  = accuracy_score(y_test, y_test_pred)
precision_test = precision_score(y_test, y_test_pred)
recall_test    = recall_score(y_test, y_test_pred)
f1_test        = f1_score(y_test, y_test_pred)

print("Test Metrics:")
print(classification_report(y_test, y_test_pred, digits=4))

Test Metrics:
              precision    recall  f1-score   support

           0     0.9627    0.8534    0.9048     50604
           1     0.4852    0.8068    0.6060      8664

    accuracy                         0.8466     59268
   macro avg     0.7239    0.8301    0.7554     59268
weighted avg     0.8929    0.8466    0.8611     59268



In [52]:
# 3. Retrieval and storage of the model's best parameters and metrics

# Parameters
best_params = grid_search.best_params_          # Only tuned parameters
full_params = best_model.get_params()           # Full pipeline configuration

# Metrics
metrics = {
    "train_accuracy":  accuracy_score(y_train, y_train_pred),
    "train_precision": precision_score(y_train, y_train_pred),
    "train_recall":    recall_score(y_train, y_train_pred),
    "train_f1":        f1_score(y_train, y_train_pred),

    "test_accuracy":   accuracy_score(y_test, y_test_pred),
    "test_precision":  precision_score(y_test, y_test_pred),
    "test_recall":     recall_score(y_test, y_test_pred),
    "test_f1":         f1_score(y_test, y_test_pred),
}

In [53]:
# Save trained model and metrics to files

joblib.dump(best_model, folder+'model.pkl')    # Trained model

with open(folder+'metrics.json', 'w') as f:    # Metrics
    json.dump(metrics, f)

## 5. Logging Metrics with MLflow

MLflow will be used to record metrics and hyperparameters from training, making it easier to manage and compare experiments and models from a centralized location.

1. **Configuration of the MLflow Server:**

    The MLflow server is initialized **from the terminal** using the command **mlflow server --host 127.0.0.1 --port 8080**, with the backend configuration and the artifacts directory. (This could also be done from a Docker Compose file.) 

2. **Configuration of each experiment:**

    The URI (Unifrom Resource Identifier) and the name of the experiment are set using `mlflow.set_tracking_uri`and `mlflow.set_experiment`, respectively.

3. **Logging of model and metrics:**

    A run of MLflow is setup with `mlflow.start_run()` and the metrics and trained model are logged using `mlflow.log_metric` and `mlflow.sklearn.log_model`.

4. **Registration of the model in the model log:**

    Using `mlflow.register_model`, the model is logged into the Registry of MLflow Models.

5. **Version management and deployment:**

    * Information about the model versions is retrieved, and the transition of versions and stages is carried out using specific functions from the MLflow API. 
    
    * The process waits until the transition to the "Staging" stage is complete before updating the model's description with `client.update_model_version`.


In [ ]:
# LOADING MODEL AND METRICS IF NEEDED
# After switching kernels to be able to work with MLflow, 
# uncomment the following lines and run this cell:

#folder = "/home/usuario/Documents/KeepCoding/MLOps-LLMOps/mlops-llmops-bd16/Practical/"

#df = pd.read_pickle(folder+"preprocessed_data.pkl")
#best_model = joblib.load(folder+"model.pkl")

#with open(folder+"metrics.json", "r") as f:
    #metrics = json.load(f)

In [54]:
# 2. Configuration of an experiment

# Notebook must point to the same tracking URI as the server
mlflow.set_tracking_uri("http://127.0.0.1:8080")  # must match MLflow server

# Set experiment name
mlflow.set_experiment('SentimentClassification_LogisticRegression')

# 3. Logging of model and metrics

with mlflow.start_run(run_name='KC-BD16'):

    # Log train metrics
    mlflow.log_metric("f1_train", metrics["train_f1"])
    mlflow.log_metric("recall_train", metrics["train_recall"])
    mlflow.log_metric("precision_train", metrics["train_precision"])
    mlflow.log_metric("accuracy_train", metrics["train_accuracy"])
    mlflow.log_metric("f1_test", metrics["test_f1"])
    mlflow.log_metric("recall_test", metrics["test_recall"])
    mlflow.log_metric("precision_test", metrics["test_precision"])
    mlflow.log_metric("accuracy_test", metrics["test_accuracy"])
    
    # Log hyperparameters
    mlflow.log_params(grid_search.best_params_)

    # Log full pipeline configuration
    mlflow.log_params({
        f"param_{k}": v
        for k, v in best_model.get_params().items()
    })

    # Log trained model
    mlflow.sklearn.log_model(best_model, "classifier")


2026/03/03 02:48:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/03 02:48:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run KC-BD16 at: http://127.0.0.1:8080/#/experiments/2/runs/9d5042c391834d6f9d24b193ddb2b46b
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/2


In [55]:
# 4. Registration of model
model_name = 'SentimentClassifier'
model_mlflow = mlflow.pyfunc.load_model(f'models:/{model_name}/Production')

## Generar .py de funciones y main con al menos dos argumentos de entrada.

In [56]:
!pip install "mlflow>=3.10.0" langchain langchain-google-genai google-generativeai python-dotenv

  Using cached langchain-1.2.10-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_google_genai-4.2.1-py3-none-any.whl.metadata (2.7 kB)
  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
  Using cached langgraph-1.0.10-py3-none-any.whl.metadata (7.4 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.8 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached langgraph_checkpoint-4.0.1-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_prebuilt-1.0.8-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.9-py3-none-any.whl.metadata (1.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached google_genai-1.65.0-py3-none-any.whl.metadata (51 kB)
  Using cached sniffio-1.3.1-py3-none-any.wh

In [57]:
import os
import mlflow
from dotenv import load_dotenv

In [59]:
load_dotenv()

# Configure tracking URI
MLFLOW_TRACKING_URI = "http://127.0.0.1:8080"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

# Configure the API key for Google
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("Configure GOOGLE_API_KEY in your .env file")

print("API key correctly configured.")

MLflow Tracking URI: http://127.0.0.1:8080
API key correctly configured.


In [ ]:
# HERE WE ARE USING SPECIFICALLY GOOGLE MODELS

import google.generativeai as genai

# Mobilize auto-tracking for Gemini
mlflow.gemini.autolog()

# Configure the experiment
mlflow.set_experiment("Gemini_Direct_Tracking")

# Configure Google SDK
experiment = genai.configure(api_key=GOOGLE_API_KEY)

print("MLflow Gemini autolog enabled.")
print(f"Experiment: {experiment}")

2026/03/03 19:57:41 INFO mlflow.tracking.fluent: Experiment with name 'Gemini_Direct_Tracking' does not exist. Creating a new experiment.


MLflow Gemini autolog enabled.
Experiment: <Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1772564261753, experiment_id='3', last_update_time=1772564261753, lifecycle_stage='active', name='Gemini_Direct_Tracking', tags={}, workspace='default'>


In [61]:
model = genai.GenerativeModel("gemini-2.5-flash")

# Start with a simple call - MLflow will automatically capture the trace
response = model.generate_content("What is MLflow and what is it used for? Answer in 2-3 sentences.")

print("Answer: ")
print(response.text)

Answer: 
MLflow is an open-source platform designed to manage the entire machine learning (ML) lifecycle. It provides tools for tracking experiments (parameters, metrics, artifacts), packaging reproducible ML code into "projects," managing and versioning models through a "registry," and deploying them to various serving environments. Its primary use is to streamline ML development, ensure reproducibility, and facilitate MLOps practices.


Trace(trace_id=tr-39258144163a9874f89075cefc265050)

In [63]:
# Example of a multi-turn chat
chat = model.start_chat(history=[])

# First question
response1 = chat.send_message("In one sentence, explain what Docker is.")
print("1st turn: ", response1.text)

# Second question
response2 = chat.send_message("And Kubernetes? Also in one sentence.")
print("2nd turn: ", response2.text)



1st turn:  Docker is a platform that packages applications and all their dependencies into standardized, isolated units called containers, ensuring they run consistently across any environment.
2nd turn:  Kubernetes is an open-source container orchestration platform that automates the deployment, scaling, and management of containerized applications across a cluster of machines.


[Trace(trace_id=tr-4e406f2f261be919410136036f4f8521), Trace(trace_id=tr-e376f4893680dfb35b52dfa25fc8675e)]

In [64]:
# Third question
response3 = chat.send_message("Explain Docker in a very detailed and technical manner.")
print("3rd turn: ", response3.text)

3rd turn:  Docker is an open-source platform designed to facilitate the development, packaging, and deployment of applications within isolated, standardized environments called **containers**, utilizing Linux kernel features like **namespaces** and **cgroups** to provide lightweight virtualization.

Here's a breakdown of its technical intricacies:

1.  **Core Abstractions:**
    *   **Docker Images:** These are read-only, immutable templates that contain an application, its dependencies, runtime, and the necessary configurations. Images are built from a `Dockerfile`, which is a plain-text file containing a series of instructions (`FROM`, `RUN`, `COPY`, `EXPOSE`, `CMD`, `ENTRYPOINT`) that define the environment layer by layer. Images are constructed using a **Union File System (UFS)** (e.g., OverlayFS, AUFS), where each instruction in the Dockerfile creates a new, immutable read-only layer. This layered architecture enables efficient storage, caching, and distribution, as common base la

Trace(trace_id=tr-a3c8886c4ac5d9ee46ed49a944848d15)

In [65]:
# Fourth question
response4 = chat.send_message("In one setence, compare how prompt_token_count, candidates_token_count and total_token_count are computed in each prompt.")
print("4th turn: ", response4.text)

4th turn:  For each prompt, `prompt_token_count` tracks the tokens in the *input query*, `candidates_token_count` tallies the tokens in the *model's generated response*, and `total_token_count` is their aggregate sum.


Trace(trace_id=tr-59e3f59e754cd55a91738872221419d4)

In [ ]:
# New experiment, with TRACING using the LANGCHAIN LIBRARY.
# LangChain is essentially a layer that allows us to 
# always use the same syntax in all our calls, 
# regardless of the selected model.


from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage

# Enable auto-tracing for LangChain

mlflow.set_experiment("LangChain_Gemini_Tracing")

print("MLflow LangChain autolog enabled.")
print(f"Experiment: LancChain_Gemini_Tracing")

2026/03/04 10:01:51 INFO mlflow.tracking.fluent: Experiment with name 'LangChain_Gemini_Tracing' does not exist. Creating a new experiment.


MLflow LangChain autolog enabled.
Experiment: LancChain_Gemini_Tracing


In [ ]:
# This script selects the model, temperature, etc.
# Just modify this script as needed.

llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    google_api_key = GOOGLE_API_KEY,
    temperature = 0.7,
    max_tokens = 1024
)

print("LangChain Model initialized.")

LangChain Model initialized.


In [ ]:
# We make a simple call, and the trace is captured automatically
#   SystemMessage defines the role (behavior) we want the model to have
#   HumanMessage is the actual prompt

messages = [
    SystemMessage(content="You are an expert in MLOps."),
    HumanMessage(content="What is 'experiment tracking' in ML?")
]

response = llm.invoke(messages)

print("Response:")
print(response.content)

## Chat Prompt Templates

Chat Prompt Templates are really the base that is used today, as they help to obtain a result in the expected structure.

It can be useful to see what system prompts (roles) are used in different Generative AI models. 
A search such as "openai prompt leaks" can yield interesting results and give ideas on how to better specify the system role and thus get results that are better aligned with one's goals.

In [70]:
# Using a prompt template
template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {topic}. Respond in a {style} manner."),
    ("human", "{question}")
])

# We create the chain prompt -> LLM -> Parser 
chain = template | llm | StrOutputParser()
print("Chain created: PromptTemplate -> LLM -> StrOutputParser")

Chain created: PromptTemplate -> LLM -> StrOutputParser


In [71]:
# Now we run the chain (MLflow is still tracing each component)

result = chain.invoke({      # chain.invoke() makes the call (similar to LLM.invoke() but with a chain)
    "topic": "kubernetes",
    "style": "concise and technical",
    "question": "What are the main components of a cluster?"
})

print("Response from the Chain:")
print(result)

Response from the Chain:
A Kubernetes cluster consists of two primary types of nodes:

*   **Control Plane Nodes:** These nodes manage the cluster's state and desired configuration. Key components include:
    *   **kube-apiserver:** Exposes the Kubernetes API.
    *   **etcd:** Consistent and highly-available key-value store used as Kubernetes' backing store for all cluster data.
    *   **kube-scheduler:** Watches for newly created Pods with no assigned node and selects a node for them to run on.
    *   **kube-controller-manager:** Runs controller processes that regulate the cluster's state.
    *   **cloud-controller-manager (optional):** Embeds cloud-specific control logic.

*   **Worker Nodes:** These nodes run the actual applications (Pods). Key components include:
    *   **kubelet:** An agent that runs on each node and ensures that containers are running in a Pod as described in the PodSpec.
    *   **kube-proxy:** A network proxy that runs on each node, maintaining network ru

Trace(trace_id=tr-06ab298c264eb390e3e01cf190c5d91e)

In [ ]:
# Diferent prompt versions for comparison purposes
prompt_versions = [
    {
        "name": "basic",
        "system": "You are a helpful assistant.",
        "user": "What is Docker?"
    },
    {
        "name": "expert",
        "system": "You are a DevOps expert with 10 years experience. Respond in a technical but accessible manner.",
        "user": "What is Docker?"
    },
    {
        "name": "structured",
        "system": "You are a DevOps expert. Respond in this format: 1) Definition, 2) Usage cases, 3) Benefits.",
        "user": "What is Docker?"
    }
]
for version in prompt_versions:
    print(f"\n{'='*50}")
    print(f"Version: {version['name']}")
    print(f"{'='*50}")
    
    messages = [
        SystemMessage(content=version["system"]),
        HumanMessage(content=version["user"])
    ]
    response = llm.invoke(messages)
    print(response.content[:300] + "...")



Version: basic
Docker is an open-source platform that **automates the deployment, scaling, and management of applications using containers.**

Think of containers as **lightweight, standalone, executable packages of software** that include everything needed to run an application: the code, runtime, system tools, s...

Version: expert
Alright, let's break down Docker from a DevOps perspective. Imagine you're building and deploying software, and you've probably run into the classic "it works on my machine" problem. You know, where your application runs perfectly on your laptop, but then when you try to run it on a staging server o...

Version: structured
Here's a breakdown of Docker from a DevOps perspective:

### 1) Definition

Docker is an open-source platform that automates the deployment, scaling, and management of applications using **containers**. A container is a lightweight, standalone, executable package of software that includes everything...


[Trace(trace_id=tr-157a516d03a508eac0ae8e8653f08f73), Trace(trace_id=tr-3286345c22cbb3fcdf1498003cd641f4), Trace(trace_id=tr-f01119081a155c585912b95eb2dfac94)]

In [73]:
import pandas as pd
import time

# Dataset de evaluación
eval_data = pd.DataFrame({
    "question": [
        "What is a Docker container?",
        "What is Kubernetes used for?",
        "What is CI/CD?"
    ],
    "expected_keywords": [
        ["software", "dependencies", "isolated"],
        ["orchestration", "containers", "scaling"],
        ["integration", "continue", "deployment"]
    ]
})

print(eval_data)

                       question                     expected_keywords
0   What is a Docker container?    [software, dependencies, isolated]
1  What is Kubernetes used for?  [orchestration, containers, scaling]
2                What is CI/CD?   [integration, continue, deployment]


In [74]:
mlflow.set_experiment("LLM_Evaluation")

with mlflow.start_run(run_name="evaluation_run"):
    results = []
    
    for idx, row in eval_data.iterrows():
        start = time.time()
        
        messages = [
            SystemMessage(content="Respond in a concise and technical manner."),
            HumanMessage(content=row["question"])
        ]
        response = llm.invoke(messages)
        
        latency = time.time() - start
        
        # Verificamos keywords presentes
        response_lower = response.content.lower()
        keywords_found = sum(1 for kw in row["expected_keywords"] if kw in response_lower)
        keyword_score = keywords_found / len(row["expected_keywords"])
        
        results.append({
            "question": row["question"],
            "response": response.content,
            "latency": latency,
            "keyword_score": keyword_score
        })
        
        mlflow.log_metric(f"latency_q{idx}", latency)
        mlflow.log_metric(f"keyword_score_q{idx}", keyword_score)
    
    # Métricas agregadas
    results_df = pd.DataFrame(results)
    mlflow.log_metric("avg_latency", results_df["latency"].mean())
    mlflow.log_metric("avg_keyword_score", results_df["keyword_score"].mean())
    
    # Guardamos resultados
    mlflow.log_table(results_df, "evaluation_results.json")
    
    print("Resultados de evaluación:")
    print(results_df[["question", "latency", "keyword_score"]])

2026/03/04 12:00:18 INFO mlflow.tracking.fluent: Experiment with name 'LLM_Evaluation' does not exist. Creating a new experiment.


Resultados de evaluación:
                       question   latency  keyword_score
0   What is a Docker container?  1.027041       0.666667
1  What is Kubernetes used for?  0.834607       0.666667
2                What is CI/CD?  1.431441       0.666667
🏃 View run evaluation_run at: http://127.0.0.1:8080/#/experiments/5/runs/638d5524d08b4337ad0add1ce1621a99
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/5


[Trace(trace_id=tr-c45cb5e70c6bf461fa3a959456be07dc), Trace(trace_id=tr-e9626d04ae2eb93509642bbf246f9515), Trace(trace_id=tr-fb2db2acefe9403fd3b906a5066ae1e5)]

## Práctica parte FastAPI

### Para esta parte de la práctica teneis que generar un script con al menos 5 modulos app.get y dos de ellos tienen que ser pipelines de HF.

### Parte de la practica se tendra que entregar en capturas de pantalla. Las capturas de pantalla a adjuntas son las siguientes.

### 1. Captura de la pantalla docs con al menos 5 modulos.
### 2. Captura de cada una de los modulos con la respuesta dentro de docs.
### 3. Captura de cada uno de los modulos en la llamada https.
### 4. Todo el codigo usado durante el proceso. Notebooks y scripts.

### Opcional

### 5. Despliegue del script en GCP Cloud Run